In [1]:
import os
import requests

In [2]:
# Load API keys
with open(r"C:\mindful-ai\sapient-ds\2024\presentation\week-06\openai-api-key-purushotham.txt") as f:
    openai_api_key = f.read().strip()
openweathermap_api_key = "94d2513070b90c306f61b4a02b3cad38" 

In [27]:
from langchain_openai import OpenAI
from langchain_core.tools import tool
from langchain.agents.react.agent import create_react_agent
from langchain.agents import AgentExecutor
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain.memory import ConversationBufferMemory

In [14]:
@tool("get_weather", return_direct=True)
def get_weather(city: str) -> str:
    """Get today's weather in a given city. Input: city name (str)."""
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": city, "appid": openweathermap_api_key, "units": "metric"}
    r = requests.get(url, params=params, timeout=10)
    if r.status_code != 200:
        return f"Error fetching weather: {r.text}"
    data = r.json()
    desc = data["weather"][0]["description"]
    temp = data["main"]["temp"]
    return f"{city}: {desc}, {temp:.1f}°C"

In [15]:
@tool("recommend_clothing", return_direct=True)
def recommend_clothing(weather: str) -> str:
    """Suggest clothing based on a weather description string."""
    advice = []
    if "rain" in weather.lower():
        advice.append("carry an umbrella")
    if "snow" in weather.lower() or "cold" in weather.lower() or "°C" in weather and float(weather.split("°C")[0].split()[-1]) < 10:
        advice.append("wear a warm coat")
    elif "°C" in weather and 10 <= float(weather.split("°C")[0].split()[-1]) < 20:
        advice.append("wear a light jacket")
    elif "°C" in weather and float(weather.split("°C")[0].split()[-1]) >= 20:
        advice.append("wear light clothes")
    return f"Clothing advice: {', '.join(advice)}"

In [16]:
@tool("search_attractions", return_direct=True)
def search_attractions(city: str) -> str:
    """Mock tool: return top attractions in a city."""
    # (in a real demo, integrate with Wikipedia API or SerpAPI)
    mock_db = {
        "Paris": ["Louvre Museum", "Eiffel Tower", "Notre Dame Cathedral", "Montmartre", "French food markets"],
        "London": ["British Museum", "Tower of London", "Camden Market", "Westminster Abbey"]
    }
    return f"Top attractions in {city}: {', '.join(mock_db.get(city, ['local sights']))}"

In [17]:
llm = OpenAI(model="gpt-4o-mini", temperature=0, openai_api_key=openai_api_key)

In [35]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [19]:
prompt = PromptTemplate.from_template(
    "You are a travel assistant with tools for weather, clothing, and attractions.\n"
    "You remember the user's interests from chat history.\n\n"
    "Chat history:\n{chat_history}\n\n"
    "User query: {input}\n"
)

In [36]:
tools = [get_weather, recommend_clothing, search_attractions]

In [44]:
# ===== 3) Custom ReAct Prompt =====
from langchain.prompts import PromptTemplate

custom_prompt = PromptTemplate(
    input_variables=["input", "tools", "tool_names", "agent_scratchpad", "chat_history"],
    template="""You are a smart travel assistant. Use the tools provided to help the user with travel planning.

TOOLS:
{tools}

You may call these tools when necessary. Always extract the city name from the conversation history below:
{chat_history}

Follow this format:

Question: the input question
Thought: reasoning about what to do
Action: the tool name, one of [{tool_names}]
Action Input: the input to the tool
Observation: result of running the tool
... (this Thought/Action/Observation may repeat)
Thought: I now know the final answer
Final Answer: the final helpful answer to the user

Begin!

Question: {input}
{agent_scratchpad}
"""
)


In [45]:

agent = create_react_agent(llm, tools, custom_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, memory=memory, verbose=True)

In [46]:
if __name__ == "__main__":
    query = "I am visiting London. What's the weather and what clothes should I pack?"
    response = agent_executor.invoke({"input": query})
    print("Agent Output:", response["output"])

    # Follow-up to test memory
    followup = "Also suggest me some tourist attractions there."
    response2 = agent_executor.invoke({"input": followup})
    print("Agent Output:", response2["output"])



> Entering new AgentExecutor chain...
Thought: I need to find out the weather in London to recommend appropriate clothing.
Action: get_weather
Action Input: LondonLondon: overcast clouds, 18.3°C


> Finished chain.
Agent Output: London: overcast clouds, 18.3°C


> Entering new AgentExecutor chain...
Thought: I need to find out what attractions are available in London.
Action: search_attractions
Action Input: LondonTop attractions in London: British Museum, Tower of London, Camden Market, Westminster Abbey


> Finished chain.
Agent Output: Top attractions in London: British Museum, Tower of London, Camden Market, Westminster Abbey
